Credits: Forked from [deep-learning-keras-tensorflow](https://github.com/leriomaggio/deep-learning-keras-tensorflow) by Valerio Maggio


# RNN using LSTM 
       



<img src="imgs/RNN-rolled.png"/ width="80px" height="80px">

<img src="imgs/RNN-unrolled.png"/ width="400px" height="400px">

<img src="imgs/LSTM3-chain.png"/ width="60%">

_source: http://colah.github.io/posts/2015-08-Understanding-LSTMs_

In [ ]:
from keras.optimizers import SGD
from keras.preprocessing.text import one_hot, text_to_word_sequence, base_filter
from keras.utils import np_utils
from keras.models import Sequential
from keras.layers.core import Dense, Dropout, Activation
from keras.layers.embeddings import Embedding
from keras.layers.recurrent import LSTM, GRU
from keras.preprocessing import sequence

### Reading blog post from data directory

Import the libraries needed for this section:
- **os**
- **pickle**
- **numpy**

In [ ]:
import os
import pickle
import numpy as np

Run this cell and inspect the output to verify the deep learning operations produce the expected results.

In [ ]:
DATA_DIRECTORY = os.path.join(os.path.abspath(os.path.curdir), 'data')
print(DATA_DIRECTORY)

The code below implements the next step in this deep learning workflow. See the inline comments for details on each operation.

In [ ]:
male_posts = []
female_post = []

The code below implements the next step in this deep learning workflow. See the inline comments for details on each operation.

In [ ]:
with open(os.path.join(DATA_DIRECTORY,"male_blog_list.txt"),"rb") as male_file:
    male_posts= pickle.load(male_file)
    
with open(os.path.join(DATA_DIRECTORY,"female_blog_list.txt"),"rb") as female_file:
    female_posts = pickle.load(female_file)

The code below implements the next step in this deep learning workflow. See the inline comments for details on each operation.

In [ ]:
filtered_male_posts = list(filter(lambda p: len(p) > 0, male_posts))
filtered_female_posts = list(filter(lambda p: len(p) > 0, female_posts))

text processing - one hot builds index of the words

The code below implements this step in the deep learning workflow.

In [ ]:
# text processing - one hot builds index of the words
male_one_hot = []
female_one_hot = []
n = 30000
for post in filtered_male_posts:
    try:
        male_one_hot.append(one_hot(post, n, split=" ", filters=base_filter(), lower=True))
    except:
        continue

for post in filtered_female_posts:
    try:
        female_one_hot.append(one_hot(post,n,split=" ",filters=base_filter(),lower=True))
    except:
        continue

0 for male, 1 for female

The code below implements this step in the deep learning workflow.

In [ ]:
# 0 for male, 1 for female
concatenate_array_rnn = np.concatenate((np.zeros(len(male_one_hot)),
                                        np.ones(len(female_one_hot))))

**Train/Test Split:** Divide your dataset into separate training and testing subsets to evaluate how well your model generalizes to unseen data.

- **Training set** (~70-80%): The model learns from this data
- **Test set** (~20-30%): Held out, only used for final evaluation

Without this split, you'd have no way to detect **overfitting** — when a model memorizes training data but fails on new data. A common split is 80/20 with `random_state` set for reproducibility.

In [ ]:
from sklearn.cross_validation import train_test_split

X_train_rnn, X_test_rnn, y_train_rnn, y_test_rnn = train_test_split(np.concatenate((female_one_hot,male_one_hot)),
                                                                    concatenate_array_rnn, 
                                                                    test_size=0.2)

Run this cell and inspect the output to verify the deep learning operations produce the expected results.

In [ ]:
maxlen = 100
X_train_rnn = sequence.pad_sequences(X_train_rnn, maxlen=maxlen)
X_test_rnn = sequence.pad_sequences(X_test_rnn, maxlen=maxlen)
print('X_train_rnn shape:', X_train_rnn.shape, y_train_rnn.shape)
print('X_test_rnn shape:', X_test_rnn.shape, y_test_rnn.shape)

**Sigmoid Function:** Squashes any real number into the range (0, 1):

$$\sigma(x) = \frac{1}{1 + e^{-x}}$$

Properties: smooth, differentiable, S-shaped curve. At $x=0$, output is 0.5.

**Used for:** Binary classification (output = probability of positive class), gating mechanisms in LSTMs/GRUs. Downside: gradients vanish for very large/small inputs (the curve flattens), which is why ReLU often replaced it in hidden layers.

In [ ]:
max_features = 30000
dimension = 128
output_dimension = 128
model = Sequential()
model.add(Embedding(max_features, dimension))
model.add(LSTM(output_dimension))
model.add(Dropout(0.5))
model.add(Dense(1))
model.add(Activation('sigmoid'))

**Mean Squared Error (MSE) Loss:** The go-to loss for **regression** — predicting continuous values:

$$\text{MSE} = \frac{1}{n} \sum_{i=1}^{n} (y_i - \hat{y}_i)^2$$

Squaring the errors means large mistakes are penalized much more than small ones. This makes MSE sensitive to outliers. For outlier-robust regression, consider MAE (Mean Absolute Error) or Huber loss.

In [ ]:
model.compile(loss='mean_squared_error', optimizer='sgd', metrics=['accuracy'])

**Model Training (.fit()):** The `.fit()` method is where the model learns from data. It adjusts the model's internal parameters to minimize prediction errors on the training data.

For different model types, `.fit()` does different things:
- **Linear models**: Finds the best-fit line/plane (minimizes squared error)
- **Decision trees**: Recursively splits data to separate classes/values
- **Neural networks**: Runs gradient descent over many epochs
- **Transformers (StandardScaler, PCA)**: Computes statistics (mean, variance, components) from the training data

In [ ]:
model.fit(X_train_rnn, y_train_rnn, batch_size=32,
          nb_epoch=4, validation_data=(X_test_rnn, y_test_rnn))

The code below implements the next step in this deep learning workflow. See the inline comments for details on each operation.

In [ ]:
score, acc = model.evaluate(X_test_rnn, y_test_rnn, batch_size=32)

Evaluate `print(score, acc)` — Jupyter renders the last expression as cell output.

In [ ]:
print(score, acc)

# Using TFIDF Vectorizer as an input instead of one hot encoder

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer

**Fit and Transform (.fit_transform()):** A convenience method that combines `.fit()` and `.transform()` in one step — learns the transformation parameters from the data and immediately applies the transformation.

**Important**: Use `.fit_transform()` only on **training data**. For test data, use `.transform()` alone to apply the same transformation learned from training. Otherwise, you leak test data statistics into the transformation ("data leakage").

In [ ]:
vectorizer = TfidfVectorizer(decode_error='ignore', norm='l2', min_df=5)
tfidf_male = vectorizer.fit_transform(filtered_male_posts)
tfidf_female = vectorizer.fit_transform(filtered_female_posts)

The code below implements the next step in this deep learning workflow. See the inline comments for details on each operation.

In [ ]:
flattened_array_tfidf_male = tfidf_male.toarray()
flattened_array_tfidf_female = tfidf_male.toarray()

The code below implements the next step in this deep learning workflow. See the inline comments for details on each operation.

In [ ]:
y_rnn = np.concatenate((np.zeros(len(flattened_array_tfidf_male)),
                                        np.ones(len(flattened_array_tfidf_female))))

**Train/Test Split:** Divide your dataset into separate training and testing subsets to evaluate how well your model generalizes to unseen data.

- **Training set** (~70-80%): The model learns from this data
- **Test set** (~20-30%): Held out, only used for final evaluation

Without this split, you'd have no way to detect **overfitting** — when a model memorizes training data but fails on new data. A common split is 80/20 with `random_state` set for reproducibility.

In [ ]:
X_train_rnn, X_test_rnn, y_train_rnn, y_test_rnn = train_test_split(np.concatenate((flattened_array_tfidf_male, 
                                                                                    flattened_array_tfidf_female)),
                                                                    y_rnn,test_size=0.2)

Run this cell and inspect the output to verify the deep learning operations produce the expected results.

In [ ]:
maxlen = 100
X_train_rnn = sequence.pad_sequences(X_train_rnn, maxlen=maxlen)
X_test_rnn = sequence.pad_sequences(X_test_rnn, maxlen=maxlen)
print('X_train_rnn shape:', X_train_rnn.shape, y_train_rnn.shape)
print('X_test_rnn shape:', X_test_rnn.shape, y_test_rnn.shape)

**Sigmoid Function:** Squashes any real number into the range (0, 1):

$$\sigma(x) = \frac{1}{1 + e^{-x}}$$

Properties: smooth, differentiable, S-shaped curve. At $x=0$, output is 0.5.

**Used for:** Binary classification (output = probability of positive class), gating mechanisms in LSTMs/GRUs. Downside: gradients vanish for very large/small inputs (the curve flattens), which is why ReLU often replaced it in hidden layers.

In [ ]:
max_features = 30000
model = Sequential()
model.add(Embedding(max_features, dimension))
model.add(LSTM(output_dimension))
model.add(Dropout(0.5))
model.add(Dense(1))
model.add(Activation('sigmoid'))

**Mean Squared Error (MSE) Loss:** The go-to loss for **regression** — predicting continuous values:

$$\text{MSE} = \frac{1}{n} \sum_{i=1}^{n} (y_i - \hat{y}_i)^2$$

Squaring the errors means large mistakes are penalized much more than small ones. This makes MSE sensitive to outliers. For outlier-robust regression, consider MAE (Mean Absolute Error) or Huber loss.

In [ ]:
model.compile(loss='mean_squared_error',optimizer='sgd', metrics=['accuracy'])

**Model Training (.fit()):** The `.fit()` method is where the model learns from data. It adjusts the model's internal parameters to minimize prediction errors on the training data.

For different model types, `.fit()` does different things:
- **Linear models**: Finds the best-fit line/plane (minimizes squared error)
- **Decision trees**: Recursively splits data to separate classes/values
- **Neural networks**: Runs gradient descent over many epochs
- **Transformers (StandardScaler, PCA)**: Computes statistics (mean, variance, components) from the training data

In [ ]:
model.fit(X_train_rnn, y_train_rnn, 
          batch_size=32, nb_epoch=4,
          validation_data=(X_test_rnn, y_test_rnn))

The code below implements the next step in this deep learning workflow. See the inline comments for details on each operation.

In [ ]:
score,acc = model.evaluate(X_test_rnn, y_test_rnn, 
                           batch_size=32)

Evaluate `print(score, acc)` — Jupyter renders the last expression as cell output.

In [ ]:
print(score, acc)

# Sentence Generation using LSTM

reading all the male text data into one string
building character set for the male posts
building two indices - character index and index of character

The code below implements this step in the deep learning workflow.

In [ ]:
# reading all the male text data into one string
male_post = ' '.join(filtered_male_posts)

#building character set for the male posts
character_set_male = set(male_post)
#building two indices - character index and index of character
char_indices = dict((c, i) for i, c in enumerate(character_set_male))
indices_char = dict((i, c) for i, c in enumerate(character_set_male))


# cut the text in semi-redundant sequences of maxlen characters
maxlen = 20
step = 1
sentences = []
next_chars = []
for i in range(0, len(male_post) - maxlen, step):
    sentences.append(male_post[i : i + maxlen])
    next_chars.append(male_post[i + maxlen])

Vectorisation of input

The code below implements this step in the deep learning workflow.

In [ ]:
#Vectorisation of input
x_male = np.zeros((len(male_post), maxlen, len(character_set_male)), dtype=np.bool)
y_male = np.zeros((len(male_post), len(character_set_male)), dtype=np.bool)

print(x_male.shape, y_male.shape)

for i, sentence in enumerate(sentences):
    for t, char in enumerate(sentence):
        x_male[i, t, char_indices[char]] = 1
    y_male[i, char_indices[next_chars[i]]] = 1

print(x_male.shape, y_male.shape)

**Softmax Function:** Converts a vector of raw scores (logits) into a **probability distribution** — all values between 0 and 1 that sum to 1:

$$\text{softmax}(z_i) = \frac{e^{z_i}}{\sum_j e^{z_j}}$$

Softmax amplifies the largest values and suppresses smaller ones. It's used as the final layer in multi-class classification: the output tells you the model's confidence for each class.

In [ ]:

# build the model: a single LSTM
print('Build model...')
model = Sequential()
model.add(LSTM(128, input_shape=(maxlen, len(character_set_male))))
model.add(Dense(len(character_set_male)))
model.add(Activation('softmax'))

optimizer = RMSprop(lr=0.01)
model.compile(loss='categorical_crossentropy', optimizer=optimizer)

**Mean Squared Error (MSE) Loss:** The go-to loss for **regression** — predicting continuous values:

$$\text{MSE} = \frac{1}{n} \sum_{i=1}^{n} (y_i - \hat{y}_i)^2$$

Squaring the errors means large mistakes are penalized much more than small ones. This makes MSE sensitive to outliers. For outlier-robust regression, consider MAE (Mean Absolute Error) or Huber loss.

In [ ]:
auto_text_generating_male_model.compile(loss='mean_squared_error',optimizer='sgd')

Import the libraries needed for this section:
- **random,**

In [ ]:
import random, sys

helper function to sample an index from a probability array

In [ ]:
# helper function to sample an index from a probability array
def sample(a, diversity=0.75):
    if random.random() > diversity:
        return np.argmax(a)
    while 1:
        i = random.randint(0, len(a)-1)
        if a[i] > random.random():
            return i

**Model Training (.fit()):** The `.fit()` method is where the model learns from data. It adjusts the model's internal parameters to minimize prediction errors on the training data.

For different model types, `.fit()` does different things:
- **Linear models**: Finds the best-fit line/plane (minimizes squared error)
- **Decision trees**: Recursively splits data to separate classes/values
- **Neural networks**: Runs gradient descent over many epochs
- **Transformers (StandardScaler, PCA)**: Computes statistics (mean, variance, components) from the training data

**Model Prediction (.predict()):** After training, `.predict()` applies the learned model to new (unseen) data to generate predictions.

- **Classification**: Returns predicted class labels
- **Regression**: Returns predicted continuous values

The quality of predictions depends on how well the model was trained and whether the new data resembles the training distribution.

In [ ]:
# train the model, output generated text after each iteration
for iteration in range(1,10):
    print()
    print('-' * 50)
    print('Iteration', iteration)
    model.fit(x_male, y_male, batch_size=128, nb_epoch=1)

    start_index = random.randint(0, len(male_post) - maxlen - 1)

    for diversity in [0.2, 0.4, 0.6, 0.8]:
        print()
        print('----- diversity:', diversity)

        generated = ''
        sentence = male_post[start_index : start_index + maxlen]
        generated += sentence
        print('----- Generating with seed: "' + sentence + '"')

        for iteration in range(400):
            try:
                x = np.zeros((1, maxlen, len(character_set_male)))
                for t, char in enumerate(sentence):
                    x[0, t, char_indices[char]] = 1.

                preds = model.predict(x, verbose=0)[0]
                next_index = sample(preds, diversity)
                next_char = indices_char[next_index]

                generated += next_char
                sentence = sentence[1:] + next_char
            except:
                continue
                
        print(sentence)
        print()